In [50]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector
import json
from pathlib import Path
from qiskit.circuit.classical import expr

In [13]:
#qc = QuantumCircuit(4)
def entangler(n):
    if n%2 == 1:
        raise("Error. Must me even number")
    qc = QuantumCircuit(n)
    for i in range(int(n/2)):
        qc.h(i)
        qc.cx(i,n-i-1)
    return qc

entangler(4).draw(output='text')

┌───┐          
q_0: ┤ H ├──■───────
     ├───┤  │       
q_1: ┤ H ├──┼────■──
     └───┘  │  ┌─┴─┐
q_2: ───────┼──┤ X ├
          ┌─┴─┐└───┘
q_3: ─────┤ X ├─────
          └───┘

In [59]:
#Commence cross-contamination
q=4
qr,cr = QuantumRegister(q),ClassicalRegister(q)
q2 = QuantumCircuit(qr,cr)
q2.compose(entangler(q),inplace=True)
q2.barrier()
q2.barrier()
q2.barrier()
#X correction
zflip(q2)
q2.barrier()
#Zcorrection
#display(Statevector(q2).draw("latex"))

q2.measure(0,0)
q2.measure(3,1)

with q2.if_test(expr.equal(cr[0], cr[1])) as else_:
    q2.store(cr[2], True)
with else_:
    q2.store(cr[2], False) 


q2.draw(output='text')


┌───┐           ░  ░  ░      ┌───┐ ░ ┌─┐    
q25_0: ┤ H ├──■────────░──░──░───■──┤ H ├─░─┤M├────
       ├───┤  │        ░  ░  ░ ┌─┴─┐└───┘ ░ └╥┘    
q25_1: ┤ H ├──┼────■───░──░──░─┤ X ├──────░──╫─────
       └───┘  │  ┌─┴─┐ ░  ░  ░ ├───┤      ░  ║     
q25_2: ───────┼──┤ X ├─░──░──░─┤ X ├──────░──╫─────
            ┌─┴─┐└───┘ ░  ░  ░ └─┬─┘┌───┐ ░  ║ ┌─┐ 
q25_3: ─────┤ X ├──────░──░──░───■──┤ H ├─░──╫─┤M├─
            └───┘      ░  ░  ░      └───┘ ░  ║ └╥┘ 
c25: 4/══════════════════════════════════════╩══╩══
                                             0  1

In [38]:
def xflip(q2):
    q2.cx(1,0)
    q2.cx(2,3)

def zflip(q2):
    q2.cx(0,1)
    q2.h(0)
    q2.cx(3,2)
    q2.h(3)